# 19 — Saving and Reloading a Trained Model (Model Persistence)

**Author:** Ali Čikić · General Assembly – Data Science Bootcamp

## 1. Goal

Hank Butler recommended: *"Save Models: Use pickling or MLflow to serialize and save fitted model objects for reuse."*

This notebook shows the basic steps:

1. train a model;
2. save the trained model to a file with Python's built-in `pickle` module;
3. load it again from the file;
4. make predictions **without training it again**;
5. check that the predictions are identical.

This is a **demonstration of saving and loading**, not a new experiment. I use the Gradient Boosting regression model
from notebook 05, with exactly the same data, features, split and settings.

**About the model:** it predicts the **recorded activity span** in days (first to last recorded event) from three
features that describe the **whole finished case**. Those features only exist after a case ends, so this model is **not**
an early-warning model, and the span is not a legal delay.

## 2. Load the Data (Same Code as Notebook 05)

In [1]:
import os
import pickle
import platform
import warnings
from pathlib import Path

os.environ["PM4PY_SHOW_PROGRESS_BAR"] = "false"  # no progress bar while loading
# hide only PM4Py's optional speed-up tip ("Install the optional requirement `r4pm` ..."); other warnings still show
warnings.filterwarnings("ignore", message="Install the optional requirement `r4pm`")

import pandas as pd
import pm4py
import sklearn
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

# the project folder: works whether Jupyter starts in the notebooks folder or in the project folder
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
data_path = project_root / "data" / "raw" / "BPIC15_1.xes"



  Welcome to PM4Py — Community Version
  Open-Source License (AGPL v3)

  📚 Docs & Examples:
     https://processintelligence.solutions/pm4py

  ⚖️  License: AGPL v3 — Commercial use requires open-sourcing your application.
     Business use without open-sourcing? A commercial license is available:
     https://processintelligence.solutions/pm4py#licensing




*When PM4Py is imported it prints a short welcome and licence message. This is normal and not an error.*

In [2]:
log = pm4py.read_xes(str(data_path))
df = pm4py.convert_to_dataframe(log)

case_data = df.groupby("case:concept:name").agg(
    events_per_case=("concept:name", "size"),
    unique_activities=("concept:name", "nunique"),
    unique_resources=("org:resource", "nunique"),
    first_event=("time:timestamp", "min"),
    last_event=("time:timestamp", "max"),
    end_date=("case:endDate", "first"),
)
case_data["duration_days"] = (case_data["last_event"] - case_data["first_event"]).dt.total_seconds() / (24 * 60 * 60)

case_data = case_data[case_data["end_date"].notna()]   # completed cases only
case_data = case_data.reset_index().rename(columns={"case:concept:name": "case_id"})
case_data = case_data[["case_id", "events_per_case", "unique_activities", "unique_resources", "duration_days"]]

print("Completed cases:", len(case_data))

Completed cases: 956


In [3]:
X = case_data[
    [
        "events_per_case",
        "unique_activities",
        "unique_resources"
    ]
]

y = case_data["duration_days"]

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training cases:", len(X_train))
print("Test cases:", len(X_test))

Training cases: 764
Test cases: 192


## 3. Train the Model (Same Settings as Notebook 05)

In [5]:
model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=2,
    random_state=42
)

model.fit(X_train, y_train)

original_pred = model.predict(X_test)

In [6]:
original_mae = mean_absolute_error(y_test, original_pred)
original_rmse = root_mean_squared_error(y_test, original_pred)
original_r2 = r2_score(y_test, original_pred)

print(f"MAE:  {original_mae:.2f} days")
print(f"RMSE: {original_rmse:.2f} days")
print(f"R²:   {original_r2:.2f}")

MAE:  51.19 days
RMSE: 104.63 days
R²:   0.49


**Check:** these must match notebook 05 (MAE 51.19, RMSE 104.63, R² 0.49).

In [7]:
reproduced = (round(original_mae, 2) == 51.19 and round(original_rmse, 2) == 104.63 and round(original_r2, 2) == 0.49)
print("Notebook 05 results reproduced:", reproduced)
assert reproduced, "The model from notebook 05 was not reproduced"

Notebook 05 results reproduced: True


## 4. What Will Be Saved

A trained model is only useful if it gets its inputs in the **same columns, in the same order**. So I save a small
dictionary with:

- the trained model;
- the feature names in the order the model expects;
- a short description (name, purpose, target, settings) and the Python and scikit-learn versions used.

**A trained model is not the training data.** The file contains what the model learned (here: 100 small trees), not the
956 cases. To make predictions later, I still need new input data with the same three features. This model needs no
separate preprocessing object (no scaler), because the trees use the three counts directly.

In [8]:
saved_object = {
    "model": model,
    "feature_names": list(X_train.columns),
    "model_name": "Gradient Boosting regression (notebook 05)",
    "purpose": "Demonstration of saving and reloading a model; whole-case regression, not an early-warning model",
    "target": "duration_days = recorded activity span in days (first to last recorded event)",
    "settings": {"n_estimators": 100, "learning_rate": 0.05, "max_depth": 2, "random_state": 42},
    "training_cases": len(X_train),
    "python_version": platform.python_version(),
    "sklearn_version": sklearn.__version__,
}

for key, value in saved_object.items():
    if key != "model":
        print(f"{key}: {value}")

feature_names: ['events_per_case', 'unique_activities', 'unique_resources']
model_name: Gradient Boosting regression (notebook 05)
purpose: Demonstration of saving and reloading a model; whole-case regression, not an early-warning model
target: duration_days = recorded activity span in days (first to last recorded event)
settings: {'n_estimators': 100, 'learning_rate': 0.05, 'max_depth': 2, 'random_state': 42}
training_cases: 764
python_version: 3.14.7
sklearn_version: 1.9.1


## 5. Save the Model with Pickle

The file goes into a `models` folder in the project. If the file **already exists, it is not overwritten**: the notebook
keeps the existing file and the next step checks that it gives the same predictions.

In [9]:
models_folder = project_root / "models"
models_folder.mkdir(exist_ok=True)
model_file = models_folder / "gradient_boosting_whole_case_regression.pkl"

if model_file.exists():
    print("The file already exists and was NOT overwritten:", model_file.name)
else:
    with open(model_file, "wb") as file:          # "wb" = write in binary mode
        pickle.dump(saved_object, file)
    print("Saved:", model_file.name)

print(f"File size: {model_file.stat().st_size / 1024:.1f} KB")

Saved: gradient_boosting_whole_case_regression.pkl
File size: 77.4 KB


## 6. Load the Model Again

**Safety:** loading a pickle file can run Python code that is hidden inside the file. So I only load the file that this
notebook created in this project's `models` folder. A pickle file from an unknown or untrusted source must never be loaded.

Loading only **reads** the trained model from the file. It does not call `fit`, so the model is **not trained again**.

In [10]:
with open(model_file, "rb") as file:              # "rb" = read in binary mode
    loaded_object = pickle.load(file)

loaded_model = loaded_object["model"]
feature_names = loaded_object["feature_names"]

print("Loaded model:", loaded_object["model_name"])
print("Feature order:", feature_names)
print("Saved with Python", loaded_object["python_version"], "and scikit-learn", loaded_object["sklearn_version"])
print("Number of trained trees inside the loaded model:", loaded_model.n_estimators_)

Loaded model: Gradient Boosting regression (notebook 05)
Feature order: ['events_per_case', 'unique_activities', 'unique_resources']
Saved with Python 3.14.7 and scikit-learn 1.9.1
Number of trained trees inside the loaded model: 100


**Versions:** a pickled scikit-learn model should be loaded with compatible Python and scikit-learn versions (ideally the
same ones). With a different version it may not load, or scikit-learn may warn that the results could differ. That is why
the versions are saved in the file and checked here.

In [11]:
print("Python now:", platform.python_version(), "| saved:", loaded_object["python_version"])
print("scikit-learn now:", sklearn.__version__, "| saved:", loaded_object["sklearn_version"])
print("Same versions:", platform.python_version() == loaded_object["python_version"]
      and sklearn.__version__ == loaded_object["sklearn_version"])

Python now: 3.14.7 | saved: 3.14.7
scikit-learn now: 1.9.1 | saved: 1.9.1
Same versions: True


## 7. Predict with the Loaded Model and Compare

The test inputs are put in the saved feature order before predicting.

In [12]:
loaded_pred = loaded_model.predict(X_test[feature_names])

largest_difference = abs(original_pred - loaded_pred).max()
print("Test cases predicted:", len(loaded_pred))
print("Predictions exactly identical:", (original_pred == loaded_pred).all())
print("Largest difference:", largest_difference, "days")

Test cases predicted: 192
Predictions exactly identical: True
Largest difference: 0.0 days


In [13]:
loaded_mae = mean_absolute_error(y_test, loaded_pred)
loaded_rmse = root_mean_squared_error(y_test, loaded_pred)
loaded_r2 = r2_score(y_test, loaded_pred)

pd.DataFrame({
    "Original model (in memory)": [original_mae, original_rmse, original_r2],
    "Reloaded model (from file)": [loaded_mae, loaded_rmse, loaded_r2],
}, index=["MAE (days)", "RMSE (days)", "R²"]).round(2)

,Original model (in memory),Reloaded model (from file)
MAE (days),51.19,51.19
RMSE (days),104.63,104.63
R²,0.49,0.49


### Using the loaded model on a new input

A new case only needs the three features, in the saved order. The values below are an invented example input, not a
real case.

In [14]:
example_case = pd.DataFrame([{"events_per_case": 45, "unique_activities": 44, "unique_resources": 3}])
example_case = example_case[feature_names]   # same column order as during training

print(f"Predicted recorded activity span: {loaded_model.predict(example_case)[0]:.1f} days")

Predicted recorded activity span: 65.3 days


## 8. Interpretation

- The Gradient Boosting model from notebook 05 was reproduced exactly (MAE 51.19 days, RMSE 104.63 days, R² 0.49) on the
  same 764 training and 192 test cases.
- The trained model, its feature order and a short description were saved in one pickle file of 77.4 KB.
- After loading, the model still contains its 100 trained trees and was **not trained again**: it was only used to predict.
- All 192 predictions of the reloaded model are **exactly identical** to the original model's (largest difference
  0.0 days), so MAE, RMSE and R² are unchanged.
- The saved and current Python and scikit-learn versions are the same (Python 3.14.7, scikit-learn 1.9.1).

So pickle can store a trained model and reuse it later without retraining. It does not make the model better: the
predictions, and their limitations, stay exactly the same.

## 9. Limitations

- Saving a model does **not** improve its predictions; it only lets me reuse the trained model without training again.
- The model uses **whole-case** features, so it is **not** an early-warning model and is not ready for real use.
- It predicts the **recorded activity span**, not a legal delay or a verified completion time.
- The 192 test cases were **already used** in notebooks 02–05 and 18.
- The saved file depends on compatible Python and scikit-learn versions, and pickle files must only be loaded from
  trusted sources.
- One Dutch municipality; not SPAR or Montenegro data.